# Experiments

In [1]:
import cptac
import cptac.utils as ut
import pandas as pd
import numpy as np
import gseapy as gp
import os
import datetime

In [2]:
TIME_START = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
STEP1_FILE = "tumor_change_20200529_195104_10000_perm.tsv"
STEP1_FILE_PATH = "step1_outputs/" + STEP1_FILE

In [3]:
# Read in the file from step 1
data = pd.read_csv(STEP1_FILE_PATH, sep='\t', index_col=0)

# Only keep genes with P values below the cutoff
data = data[data["reject_null"]]

In [4]:
# In the omics data, make all increases +1 and decreases -1
# Because these are ratioed abundances, we can't compare magnitudes between
# genes--we can only compare positive or negative
data = data.assign(simplified_change=data["change"])

data.loc[data["change"] > 0, "simplified_change"] = 1
data.loc[data["change"] < 0, "simplified_change"] = -1
data.loc[data["change"] == 0, "simplified_change"] = 0

In [18]:
import requests

def get_resp(pathway_ids, quiet=False):
    """Query the Reactome REST API to get a list of proteins contained in a particular pathway.

    Parameters:
    pathway_id (str): The pathway to get the contained proteins for.

    Returns:
    list: The proteins contained in the pathway.
    """
    # Process string input
    if isinstance(pathway_ids, str):
        pathway_ids = [pathway_ids]

    # Set headers and url
    headers = {"accept": "application/json"}

    # Loop over ids and get the interacting pathways
    all_pathway_df = pd.DataFrame()
    for pathway_id in pathway_ids:

        # Send the request
        url = f"https://reactome.org/ContentService/data/participants/{pathway_id}"
        resp = requests.get(url, headers=headers)

        if resp.status_code == 404 or (resp.status_code == requests.codes.ok and (len(resp.content.decode("utf-8")) == 0 or len(resp.json()) == 0)):
            if not quiet:
                warnings.warn(f"The query for '{pathway_id}' found no results. You may have mistyped the pathway ID.", ParameterWarning, stacklevel=2)
            continue
        elif resp.status_code != requests.codes.ok:
            raise HttpResponseError(f"Your query returned an HTTP status {resp.status_code}. The content returned from the request may be helpful:\n{resp.content.decode('utf-8')}")
        
        return resp

In [19]:
resp = get_resp("R-HSA-9006934")
# resp = search_reactome_proteins_in_pathways("R-HSA-2404192")

In [20]:
resp.json()

[{'peDbId': 9031773,
  'displayName': 'PI3K:p-KIT:sSCF dimer:p-KIT,PI3K:p(Y)-GAB2:GRB2:p-SHP2:p-KIT complex [plasma membrane]',
  'schemaClass': 'DefinedSet',
  'refEntities': [{'identifier': 'P06241',
    'schemaClass': 'ReferenceGeneProduct',
    'displayName': 'UniProt:P06241 FYN',
    'dbId': 55160,
    'icon': 'EntityWithAccessionedSequence',
    'url': 'https://purl.uniprot.org/uniprot/P06241'},
   {'identifier': 'P06239',
    'schemaClass': 'ReferenceGeneProduct',
    'displayName': 'UniProt:P06239 LCK',
    'dbId': 58457,
    'icon': 'EntityWithAccessionedSequence',
    'url': 'https://purl.uniprot.org/uniprot/P06239'},
   {'identifier': 'P12931-1',
    'schemaClass': 'ReferenceIsoform',
    'displayName': 'UniProt:P12931-1 SRC',
    'dbId': 65044,
    'icon': 'EntityWithAccessionedSequence',
    'url': 'https://purl.uniprot.org/uniprot/P12931-1'},
   {'identifier': 'P07947',
    'schemaClass': 'ReferenceGeneProduct',
    'displayName': 'UniProt:P07947 YES1',
    'dbId': 67748,

In [21]:
upper = pd.json_normalize(resp.json())

In [22]:
upper

,peDbId,displayName,schemaClass,refEntities
0,9031773,"PI3K:p-KIT:sSCF dimer:p-KIT,PI3K:p(Y)-GAB2:GRB...",DefinedSet,"[{'identifier': 'P06241', 'schemaClass': 'Refe..."
1,179856,"PI(4,5)P2 [plasma membrane]",SimpleEntity,"[{'identifier': '18348', 'schemaClass': 'Refer..."
2,113592,ATP [cytosol],SimpleEntity,"[{'identifier': '30616', 'schemaClass': 'Refer..."
3,29370,ADP [cytosol],SimpleEntity,"[{'identifier': '456216', 'schemaClass': 'Refe..."
4,179838,"PI(3,4,5)P3 [plasma membrane]",SimpleEntity,"[{'identifier': '16618', 'schemaClass': 'Refer..."
5,205271,SFKs:p-KIT complex [plasma membrane],Complex,"[{'identifier': 'P06241', 'schemaClass': 'Refe..."
6,210620,JAK2 [cytosol],EntityWithAccessionedSequence,"[{'identifier': 'O60674', 'schemaClass': 'Refe..."
7,1433493,JAK2:SFKS:p-KIT complex [plasma membrane],Complex,"[{'identifier': 'P06241', 'schemaClass': 'Refe..."
8,1433545,GAB2:GRB2:p-SHP2:p-KIT complex [plasma membrane],Complex,"[{'identifier': 'Q9UQC2', 'schemaClass': 'Refe..."
9,1562560,p(Y)-GAB2:GRB2:p-SHP2:p-KIT complex [plasma me...,Complex,"[{'identifier': 'P06241', 'schemaClass': 'Refe..."


In [24]:
pd.options.display.max_colwidth = None

In [25]:
# For a get_members_info function?
out = pd.json_normalize(
    resp.json(),
    record_path=["refEntities"],
    meta=["peDbId", "displayName", "schemaClass"],
    meta_prefix="parent_",
)
out

,identifier,schemaClass,displayName,dbId,icon,url,parent_peDbId,parent_displayName,parent_schemaClass
0,P06241,ReferenceGeneProduct,UniProt:P06241 FYN,55160,EntityWithAccessionedSequence,https://purl.uniprot.org/uniprot/P06241,9031773,"PI3K:p-KIT:sSCF dimer:p-KIT,PI3K:p(Y)-GAB2:GRB2:p-SHP2:p-KIT complex [plasma membrane]",DefinedSet
1,P06239,ReferenceGeneProduct,UniProt:P06239 LCK,58457,EntityWithAccessionedSequence,https://purl.uniprot.org/uniprot/P06239,9031773,"PI3K:p-KIT:sSCF dimer:p-KIT,PI3K:p(Y)-GAB2:GRB2:p-SHP2:p-KIT complex [plasma membrane]",DefinedSet
2,P12931-1,ReferenceIsoform,UniProt:P12931-1 SRC,65044,EntityWithAccessionedSequence,https://purl.uniprot.org/uniprot/P12931-1,9031773,"PI3K:p-KIT:sSCF dimer:p-KIT,PI3K:p(Y)-GAB2:GRB2:p-SHP2:p-KIT complex [plasma membrane]",DefinedSet
3,P07947,ReferenceGeneProduct,UniProt:P07947 YES1,67748,EntityWithAccessionedSequence,https://purl.uniprot.org/uniprot/P07947,9031773,"PI3K:p-KIT:sSCF dimer:p-KIT,PI3K:p(Y)-GAB2:GRB2:p-SHP2:p-KIT complex [plasma membrane]",DefinedSet
4,P07948,ReferenceGeneProduct,UniProt:P07948 LYN,58853,EntityWithAccessionedSequence,https://purl.uniprot.org/uniprot/P07948,9031773,"PI3K:p-KIT:sSCF dimer:p-KIT,PI3K:p(Y)-GAB2:GRB2:p-SHP2:p-KIT complex [plasma membrane]",DefinedSet
5,P10721,ReferenceGeneProduct,UniProt:P10721 KIT,58103,EntityWithAccessionedSequence,https://purl.uniprot.org/uniprot/P10721,9031773,"PI3K:p-KIT:sSCF dimer:p-KIT,PI3K:p(Y)-GAB2:GRB2:p-SHP2:p-KIT complex [plasma membrane]",DefinedSet
6,P21583-1,ReferenceIsoform,UniProt:P21583-1 KITLG,403295,EntityWithAccessionedSequence,https://purl.uniprot.org/uniprot/P21583-1,9031773,"PI3K:p-KIT:sSCF dimer:p-KIT,PI3K:p(Y)-GAB2:GRB2:p-SHP2:p-KIT complex [plasma membrane]",DefinedSet
7,Q06124,ReferenceGeneProduct,UniProt:Q06124 PTPN11,62439,EntityWithAccessionedSequence,https://purl.uniprot.org/uniprot/Q06124,9031773,"PI3K:p-KIT:sSCF dimer:p-KIT,PI3K:p(Y)-GAB2:GRB2:p-SHP2:p-KIT complex [plasma membrane]",DefinedSet
8,P62993-1,ReferenceIsoform,UniProt:P62993-1 GRB2,55928,EntityWithAccessionedSequence,https://purl.uniprot.org/uniprot/P62993-1,9031773,"PI3K:p-KIT:sSCF dimer:p-KIT,PI3K:p(Y)-GAB2:GRB2:p-SHP2:p-KIT complex [plasma membrane]",DefinedSet
9,P42336,ReferenceGeneProduct,UniProt:P42336 PIK3CA,61074,EntityWithAccessionedSequence,https://purl.uniprot.org/uniprot/P42336,9031773,"PI3K:p-KIT:sSCF dimer:p-KIT,PI3K:p(Y)-GAB2:GRB2:p-SHP2:p-KIT complex [plasma membrane]",DefinedSet
